# Phase 6 — Read-Only Dashboard

This dashboard subscribes to MQTT topics and visualizes live simulation data.

Read-only behavior:
- Subscribes to state, match, and switch topics
- Computes live metrics
- Renders map markers when `anymap-ts` is available
- Shows profile fields (`height_cm`, `hair_color_index`) in marker popup
- Does not publish control commands

In [1]:
from datetime import datetime, timezone
import json
from pathlib import Path
import sys
import time

cwd = Path.cwd()
candidate_paths = [cwd / "src", cwd.parent / "src"]
for path in candidate_paths:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

from IPython.display import display
from simulated_city.config import load_config
from simulated_city.mqtt import MqttConnector
import yaml

In [2]:
app_cfg = load_config()

config_path = Path("config.yaml")
if not config_path.exists():
    config_path = Path("../config.yaml")

raw_cfg = {}
if config_path.exists():
    raw_cfg = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}

sim_cfg = raw_cfg.get("dating_simulation", {})
BAR_IDS = sim_cfg.get("bar_ids", ["bar_1", "bar_2", "bar_3", "bar_4"])

TOPIC_BASE = app_cfg.mqtt.base_topic
TOPIC_STATE = f"{TOPIC_BASE}/agents/state"
TOPIC_MATCH = f"{TOPIC_BASE}/events/match"
TOPIC_SWITCH = f"{TOPIC_BASE}/events/switch"

print({
    "topic_state": TOPIC_STATE,
    "topic_match": TOPIC_MATCH,
    "topic_switch": TOPIC_SWITCH,
    "bars": BAR_IDS,
})

{'topic_state': 'simulated-city/agents/state', 'topic_match': 'simulated-city/events/match', 'topic_switch': 'simulated-city/events/switch', 'bars': ['bar_1', 'bar_2', 'bar_3', 'bar_4']}


In [3]:
map_widget = None
marker_ids = set()

try:
    from simulated_city.maplibre_live import LiveMapLibreMap
    map_widget = LiveMapLibreMap(center=(12.5588, 55.6699), zoom=15.9, height="550px", width="100%")
    display(map_widget)
except Exception as exc:
    print(f"Map disabled ({exc})")

In [4]:
latest_people = {}
match_reason_by_person = {}
match_reason_by_person_id = {}
match_reason_totals = {"both": 0, "height": 0, "hair": 0, "unknown": 0}
match_events_seen = 0
switch_events_seen = 0
reason_legend_id = "match-reason-legend"
reason_legend_added = False
reason_order = ["both", "height", "hair", "unknown"]
reason_colors = ["#16a34a", "#eab308", "#2563eb", "#6b7280"]

BAR_SIZE_M = float(sim_cfg.get("bar_size_m", 100.0))

# Square corners provided by you (clockwise): 1 -> 2 -> 3 -> 4
AREA_P1 = (12.556126, 55.667977)
AREA_P2 = (12.558630, 55.665437)
AREA_P3 = (12.563293, 55.667838)
AREA_P4 = (12.559868, 55.669848)

def parse_timestamp(value):
    if not value or not isinstance(value, str):
        return None
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return None

def latest_people_state():
    return dict(latest_people)

def _bilinear_point(u, v, p00, p10, p11, p01):
    lng = (
        (1.0 - u) * (1.0 - v) * p00[0]
        + u * (1.0 - v) * p10[0]
        + u * v * p11[0]
        + (1.0 - u) * v * p01[0]
    )
    lat = (
        (1.0 - u) * (1.0 - v) * p00[1]
        + u * (1.0 - v) * p10[1]
        + u * v * p11[1]
        + (1.0 - u) * v * p01[1]
    )
    return lng, lat

def xy_to_square_lnglat(x, y, width=100.0):
    u = max(0.0, min(1.0, x / width))
    v = max(0.0, min(1.0, y / width))
    return _bilinear_point(u, v, AREA_P1, AREA_P2, AREA_P3, AREA_P4)

def marker_color_for_state(state: str) -> str:
    if state == "matched":
        return "green"
    if state == "talking":
        return "yellow"
    return "blue"

def normalize_match_basis(value):
    reason = str(value or "unknown").lower()
    if reason not in reason_order:
        return "unknown"
    return reason

def reason_for_person(bar_id, person_id):
    person_key = (str(bar_id or ""), str(person_id))
    reason = match_reason_by_person.get(person_key)
    if reason is None:
        reason = match_reason_by_person_id.get(str(person_id), "unknown")
    return normalize_match_basis(reason)

def update_reason_legend(snapshot):
    global reason_legend_added
    if map_widget is None:
        return

    labels = [f"{reason}: {match_reason_totals[reason]}" for reason in reason_order]

    try:
        if reason_legend_added:
            try:
                map_widget.remove_legend(reason_legend_id)
            except Exception:
                pass

        map_widget.add_legend(
            title="Matches by reason",
            labels=labels,
            colors=reason_colors,
            position="top-left",
            legend_id=reason_legend_id,
        )
        reason_legend_added = True
    except Exception as exc:
        print(f"Legend update failed: {exc}")

def refresh_map(snapshot=None):
    global map_widget, marker_ids
    if map_widget is None:
        return

    if snapshot is None:
        snapshot = latest_people_state()

    active_marker_ids = set()
    for pid, data in snapshot.items():
        state = data.get("state", "idle")
        if state == "removed":
            continue

        lng, lat = xy_to_square_lnglat(
            data.get("x", 0.0),
            data.get("y", 0.0),
            BAR_SIZE_M,
        )
        color = marker_color_for_state(state)

        bar_id = data.get("bar_id", "")
        person_value = data.get("person_id", pid)
        match_basis_text = reason_for_person(bar_id, person_value)

        popup = (
            f"{pid} | {state} | color={data.get('color', '?')} "
            f"| height_cm={data.get('height_cm', '?')} "
            f"| hair_index={data.get('hair_color_index', '?')} "
            f"| match_basis={match_basis_text}"
        )
        marker_id = f"p-{pid}"
        active_marker_ids.add(marker_id)

        map_widget.move_marker(
            marker_id,
            (lng, lat),
            color=color,
            popup=popup,
        )

    stale_marker_ids = marker_ids - active_marker_ids
    for marker_id in stale_marker_ids:
        map_widget.remove_marker(marker_id)

    marker_ids = active_marker_ids
    update_reason_legend(snapshot)

def summary(snapshot):
    states = [p.get("state", "idle") for p in snapshot.values()]
    present = sum(1 for state in states if state != "removed")
    matched = sum(1 for state in states if state == "matched")
    return {
        "present": present,
        "matched": matched,
        "matches": match_events_seen,
        "switches": switch_events_seen,
        "events_seen": len(snapshot),
    }

def print_dashboard_summary():
    snapshot = latest_people_state()
    kpi = summary(snapshot)
    print(
        f"present={kpi['present']} matched={kpi['matched']} matches={kpi['matches']} switches={kpi['switches']} events={kpi['events_seen']}",
        flush=True,
    )

In [5]:
if "connector" in globals() and connector is not None:
    try:
        connector.disconnect()
    except Exception:
        pass

connector = MqttConnector(app_cfg.mqtt, client_id_suffix="dashboard")

def on_message(client, userdata, message):
    global match_events_seen, switch_events_seen

    try:
        payload = json.loads(message.payload.decode("utf-8"))
    except Exception:
        return

    if message.topic == TOPIC_STATE:
        person_id = payload.get("person_id")
        if person_id is None:
            return
        person_key = str(person_id)
        if payload.get("state") == "removed":
            latest_people.pop(person_key, None)
            bar_id = str(payload.get("bar_id", ""))
            reason_key = (bar_id, person_key)
            match_reason_by_person.pop(reason_key, None)
            match_reason_by_person_id.pop(person_key, None)
        else:
            latest_people[person_key] = payload
    elif message.topic == TOPIC_MATCH:
        bar_id = str(payload.get("bar_id", ""))
        person_a_id = payload.get("person_a_id")
        person_b_id = payload.get("person_b_id")
        match_basis = normalize_match_basis(payload.get("match_basis", "unknown"))

        if person_a_id is not None:
            person_a_key = str(person_a_id)
            match_reason_by_person[(bar_id, person_a_key)] = match_basis
            match_reason_by_person_id[person_a_key] = match_basis
        if person_b_id is not None:
            person_b_key = str(person_b_id)
            match_reason_by_person[(bar_id, person_b_key)] = match_basis
            match_reason_by_person_id[person_b_key] = match_basis

        match_reason_totals[match_basis] += 1
        match_events_seen += 1
        print(f"[dashboard-match] basis={match_basis} totals={match_reason_totals}")

        try:
            update_reason_legend(latest_people_state())
        except Exception as exc:
            print(f"Legend refresh failed: {exc}")
    elif message.topic == TOPIC_SWITCH:
        switch_events_seen += 1

connector.client.on_message = on_message

connector.connect()
if connector.wait_for_connection(timeout=3.0):
    connector.client.subscribe(TOPIC_STATE)
    connector.client.subscribe(TOPIC_MATCH)
    connector.client.subscribe(TOPIC_SWITCH)
    print("Dashboard subscribed to state/match/switch topics.")
else:
    print("Dashboard did not connect in time.")

Dashboard subscribed to state/match/switch topics.


In [ ]:
DASHBOARD_RUN_SECONDS = 300
REFRESH_EVERY_SECONDS = 1

run_started = time.monotonic()
while (time.monotonic() - run_started) < DASHBOARD_RUN_SECONDS:
    time.sleep(REFRESH_EVERY_SECONDS)
    print_dashboard_summary()
    refresh_map()

print("Dashboard 5-minute run complete.")

present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
present=46 matched=0 matches=0 switches=0 events=46
[dashboard-match] basis=height totals={'both': 0, 'height': 1, 'hair': 0, 'unknown': 0}
present=46 matched=2 matches=1 switches=0 events=46
present=46 matched=2 matches=1 switches=0 events=46
present=46 matched=2 matches=1 switches=0 events=46
present=46 matched=2 matches=1 switches=0 events=46
present=46 matched=2 matches

In [ ]:
connector.disconnect()
print("Dashboard disconnected.")

Dashboard disconnected.
